In [1]:
import sys
import pandas as pd
from pathlib import Path

# This moves from notebooks/ up one level to project root
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)


Using project root: C:\Users\nathan.taylor\Jupyter Notebooks\outlier_pipeline


In [2]:
from src.runner import run_all_jobs

CONFIG_PATH = PROJECT_ROOT / "configs" / "jobs_022626.yml"

results = run_all_jobs(str(CONFIG_PATH))


2026-02-26 10:52:17,575 [INFO] src.runner - Starting job: ATT_MOB_hold_time
2026-02-26 10:52:17,589 [INFO] pyhive.presto - SELECT 
--CAST(week_stop_date AS DATE) AS week_,
--date,
expert_id,
-- year_month,
metric,
icp_client,
--tenure_group,
site,
SUM(numerator) AS num,
SUM(denominator) AS den,
ROUND(
COALESCE(
CAST(SUM(numerator) AS DOUBLE) /
NULLIF(CAST(SUM(denominator) AS DOUBLE), 0.0),
0.000
),
3
) AS calc
FROM 
hive.care.expert_performance_metrics a
LEFT OUTER JOIN 
hive.care.l4_asurion_umt_ppx_pay_calendar d ON a."date" = CAST(d.event_date AS DATE)
WHERE 
LOWER(metric) = 'hold time'
AND icp_client = 'MOB-AT&T'
AND a."date" between DATE '2026-02-13' and DATE '2026-02-27' 
 --and expert_id in ()
GROUP BY 1, 2, 3, 4


2026-02-26 10:52:20,788 [INFO] src.runner - [ATT_MOB_hold_time] Raw query rows: 623
2026-02-26 10:52:20,807 [INFO] src.runner - [ATT_MOB_hold_time] After validity filters rows: 615
2026-02-26 10:52:20,812 [INFO] src.runner - [ATT_MOB_hold_time] After eligibility filter

In [8]:
print("\n===== JOB SUMMARY =====\n")

summary_df = pd.DataFrame([
    {
        "job_name": job_name,
        "rows_raw": meta.get("rows_raw"),
        "rows_valid": meta.get("rows_valid"),
        "rows_eligible": meta.get("rows_eligible"),
        "rows_out": meta.get("rows_out"),
        "status": meta.get("status"),
        "output_file": meta.get("out_path"),
    }
    for job_name, meta in results.items()
])

summary_df



===== JOB SUMMARY =====



,job_name,rows_raw,rows_valid,rows_eligible,rows_out,status,output_file
0,ATT_MOB_hold_time,623,615,516,130,success,None
1,ATT_MOB_talk_time,623,615,516,130,success,None
2,ATT_MOB_smart_offer,707,573,573,93,success,None
3,ATT_MOB_OCR,585,535,490,120,success,None
4,ATT_MOB_Transfers,623,502,479,63,success,None
5,Mcafee_CRT,162,159,159,123,success,None
6,Mcafee_Cancel,162,156,156,29,success,None
7,VZW_MOB_hold_time,474,462,426,107,success,None
8,VZW_MOB_talk_time,474,462,426,107,success,None
9,VZW_MOB_smart_offer,520,454,454,56,success,None
